In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install menpo
!pip install fvcore
!apt-get install -y p7zip-full
!git clone https://github.com/im-xiaoming/xiaoying.git
!git clone https://github.com/im-xiaoming/firework.git

In [ ]:
!7z x /content/drive/MyDrive/datasets/faces_extracted.zip -o/content/data -y
!7z x /content/drive/MyDrive/datasets/archive.zip -o/content/data -y
!7z x /content/drive/MyDrive/datasets/VN-celeb.zip -o/content/data -y

In [ ]:
import torch
import os
import torch.nn as nn
import torchvision.transforms as transforms
from xiaoying.utils import get_loader, get_model, get_head, get_optimizer, set_lr, get_val_loader, \
    get_criterion
from xiaoying import utils
from xiaoying.trainer import Trainer, TrainingArgs, get_metrics

In [ ]:
TRANSFORMS = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])


DATA_PATH = '/content/data/faces_extracted'
ACCURACY_VAL_PATH = ''
CHECKPOINT = '/content/drive/MyDrive/ir_50_checkpoint_3.pth'
SAVE_CHECKPOINT_DIR = '/content/drive/MyDrive'

BATCH_SIZE = 256
MODEL_NAME = 'ir_50' # ir_18, ir_50, vit
LEARNING_RATE = 0.1 # 0.01, 0.001, 0.0001
EPOCHS = 5

# Augmentation
low_res_augmentation_prob = 0.1
crop_augmentation_prob = 0.0
photometric_augmentation_prob = 0.1


# Head params
m = 0.4 # 0.3, 0.5,...
h = 0.333
s = 64
t_alpha = 0.99
CLASS_NUM = len(os.listdir(DATA_PATH))
OUTPUT_SIZE = 512 # 768, 1024...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
criterion = get_criterion('softmax')

dataloader = get_loader(data_path=DATA_PATH, transforms=TRANSFORMS, batch_size=BATCH_SIZE, shuffle=True,
                        low_res_augmentation_prob=low_res_augmentation_prob,
                        crop_augmentation_prob=crop_augmentation_prob,
                        photometric_augmentation_prob=photometric_augmentation_prob)
val_loader = get_val_loader(ACCURACY_VAL_PATH)

model = get_model(MODEL_NAME, device)
head = get_head(device, OUTPUT_SIZE, CLASS_NUM, m, h, s, t_alpha)
optimizer = get_optimizer(model=model, model_name=MODEL_NAME, head=head, lr=LEARNING_RATE, opt_type='SGD') # opt_type: ['sgd', 'adamw']

In [ ]:
accuracy_metric = get_metrics('accuracy')
metrics = [
    {
        'metric_name': 'accuracy',
        'metric': accuracy_metric
    }
]

In [ ]:
args = TrainingArgs(
    model_name=MODEL_NAME,
    device=device,
    epochs=EPOCHS
)

trainer = Trainer(args, dataloader, model, head, optimizer, criterion)
trainer._load_checkpoint(CHECKPOINT)
set_lr(optimizer, 0.01)
trainer.train(save_dir=SAVE_CHECKPOINT_DIR, metrics=metrics)

In [ ]:
utils.free_memory()